# MG-GAT Training

Upload as Kaggle Dataset (`mggat-pa-data`):
- `pa_reviews_train.csv`, `pa_reviews_val.csv`, `pa_reviews_test.csv`
- `pa_users_feat.csv` (without friends, ~62MB)
- `pa_biz_features.csv`
- `usr2idx.pkl`, `biz2idx.pkl`
- `user_graph.npz`, `biz_graph_geo.npz`, `biz_graph_covisit.npz`, `biz_graph_cat.npz`
- `Su_imp.npy`, `Sb_imp.npy`

To resume after session expiry:
1. Save Version after run
2. Add previous Output as Input dataset named `mggat-ckpt`
3. Run All — checkpoint auto-recovered

## Imports

In [18]:
import os, math, pickle, shutil, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

DATA_DIR  = '/kaggle/input/datasets/kikichen6868/mggat-pa-data'

WORK_DIR  = '/kaggle/working'

Device: cuda


## Model

In [19]:
class MGGAT(nn.Module):
    def __init__(self, su, sb, n_users, n_biz, n_biz_graphs=3,
                 d0_u=64, d0_b=64, d1_u=128, d1_b=128, kf=64,
                 actv1='elu', actv2='relu', r_min=1.0, r_max=5.0):
        super().__init__()
        self.kf, self.r_min, self.r_max = kf, r_min, r_max
        self.d0_u, self.d0_b = d0_u, d0_b

        self.W1_u = nn.Linear(su, d0_u, bias=False)
        self.W1_b = nn.Linear(sb, d0_b, bias=False)

        self.a_u = nn.Parameter(torch.empty(2 * d0_u))
        self.a_b = nn.Parameter(torch.empty(2 * d0_b))
        nn.init.xavier_uniform_(self.a_u.unsqueeze(0))
        nn.init.xavier_uniform_(self.a_b.unsqueeze(0))
        self.omega = nn.Parameter(torch.ones(n_biz_graphs) / n_biz_graphs)
        self.leaky_relu = nn.LeakyReLU(0.2)

        self.W2_u  = nn.Linear(d0_u, d1_u, bias=False)
        self.W2_us = nn.Linear(su, d1_u, bias=False)
        self.b1_u  = nn.Parameter(torch.zeros(d1_u))
        self.W2_b  = nn.Linear(d0_b, d1_b, bias=False)
        self.W2_bs = nn.Linear(sb, d1_b, bias=False)
        self.b1_b  = nn.Parameter(torch.zeros(d1_b))

        self.W3_u = nn.Linear(d1_u, kf, bias=False)
        self.W3_b = nn.Linear(d1_b, kf, bias=False)
        self.H4_u = nn.Embedding(n_users, kf)
        self.H4_b = nn.Embedding(n_biz, kf)
        nn.init.normal_(self.H4_u.weight, std=0.01)
        nn.init.normal_(self.H4_b.weight, std=0.01)

        self.bias_u = nn.Embedding(n_users, 1)
        self.bias_b = nn.Embedding(n_biz, 1)
        self.bias_global = nn.Parameter(torch.zeros(1))
        nn.init.zeros_(self.bias_u.weight)
        nn.init.zeros_(self.bias_b.weight)

        activations = {'elu': nn.ELU(), 'relu': nn.ReLU(), 'tanh': nn.Tanh(),
                       'selu': nn.SELU(), 'sigmoid': nn.Sigmoid(), 'linear': nn.Identity()}
        self.actv1 = activations[actv1]
        self.actv2 = activations[actv2]

    def _softmax_by_dst(self, e, dst, N):
        e_max = torch.full((N,), float('-inf'), device=e.device)
        e_max.scatter_reduce_(0, dst, e, reduce='amax', include_self=True)
        e_exp = torch.exp(e - e_max[dst])
        e_sum = torch.zeros(N, device=e.device).scatter_add_(0, dst, e_exp)
        return e_exp / (e_sum[dst] + 1e-16)

    def _compute_nig_user(self, H1_u, edge_index_u):
        src, dst = edge_index_u[0], edge_index_u[1]
        # Memory-efficient: avoid h_cat, compute dot product directly
        a_dst, a_src = self.a_u[:self.d0_u], self.a_u[self.d0_u:]
        e = (H1_u[dst] * a_dst).sum(1) + (H1_u[src] * a_src).sum(1)
        e = self.leaky_relu(e)
        return self._softmax_by_dst(e, dst, H1_u.shape[0])

    def _compute_nig_biz(self, H1_b, edge_indices_b):
        omega = torch.softmax(self.omega, dim=0)
        N = H1_b.shape[0]
        a_dst, a_src = self.a_b[:self.d0_b], self.a_b[self.d0_b:]
        all_src, all_dst, all_e = [], [], []
        for g, edge_index in enumerate(edge_indices_b):
            s, d = edge_index[0], edge_index[1]
            e_g = (H1_b[d] * a_dst).sum(1) + (H1_b[s] * a_src).sum(1)
            all_src.append(s)
            all_dst.append(d)
            all_e.append(omega[g] * e_g)
        all_src = torch.cat(all_src)
        all_dst = torch.cat(all_dst)
        all_e = self.leaky_relu(torch.cat(all_e))
        alpha = self._softmax_by_dst(all_e, all_dst, N)
        return all_src, all_dst, alpha

    def _aggregate(self, H1, alpha, src, dst, N):
        weighted = H1[src] * alpha.unsqueeze(1)
        H2 = torch.zeros(N, H1.shape[1], device=H1.device)
        H2.scatter_add_(0, dst.unsqueeze(1).expand_as(weighted), weighted)
        return H2

    def forward(self, S_u, S_b, edge_index_u, edge_indices_b, user_idx, biz_idx):
        n_u, n_b = S_u.shape[0], S_b.shape[0]
        H1_u = self.W1_u(S_u)
        H1_b = self.W1_b(S_b)

        alpha_u = self._compute_nig_user(H1_u, edge_index_u)
        src_b, dst_b, alpha_b = self._compute_nig_biz(H1_b, edge_indices_b)

        src_u, dst_u = edge_index_u[0], edge_index_u[1]
        H2_u = self._aggregate(H1_u, alpha_u, src_u, dst_u, n_u)
        H2_b = self._aggregate(H1_b, alpha_b, src_b, dst_b, n_b)

        H3_u = self.actv1(self.W2_u(H2_u) + self.W2_us(S_u) + self.b1_u)
        H3_b = self.actv1(self.W2_b(H2_b) + self.W2_bs(S_b) + self.b1_b)

        U_all = self.actv2(self.W3_u(H3_u)) + self.H4_u.weight
        B_all = self.actv2(self.W3_b(H3_b)) + self.H4_b.weight

        U_q, B_q = U_all[user_idx], B_all[biz_idx]
        logit = (U_q * B_q).sum(dim=1) + self.bias_u(user_idx).squeeze(1) \
              + self.bias_b(biz_idx).squeeze(1) + self.bias_global
        pred = (self.r_max - self.r_min) * torch.sigmoid(logit) + self.r_min
        return pred, U_all, B_all


def graph_laplacian_reg(H4, edge_index, theta2, chunk_size=200000):
    """Chunked to avoid materializing full E*kf diff tensor."""
    src, dst = edge_index[0], edge_index[1]
    E = src.shape[0]
    reg = torch.tensor(0.0, device=H4.device)
    for i in range(0, E, chunk_size):
        s, d = src[i:i+chunk_size], dst[i:i+chunk_size]
        diff = H4[s] - H4[d]
        reg = reg + (diff * diff).sum()
    return 0.5 * reg + theta2 * (H4 * H4).sum()

print('Model defined (memory-optimized).')

Model defined (memory-optimized).


## Data Loading

In [20]:
def build_feature_matrix(df_feat, id_col, idx_map, feat_cols):
    """Vectorized: map DataFrame rows to feature matrix by idx_map order."""
    n = len(idx_map)
    mat = np.zeros((n, len(feat_cols)), dtype=np.float32)
    df = df_feat[df_feat[id_col].isin(idx_map)].copy()
    df['_idx'] = df[id_col].map(idx_map)
    df = df.dropna(subset=['_idx'])
    df['_idx'] = df['_idx'].astype(int)
    mat[df['_idx'].values] = df[feat_cols].values.astype(np.float32)
    return mat


def load_data():
    print('Loading data...')
    with open(os.path.join(DATA_DIR, 'usr2idx.pkl'), 'rb') as f: usr2idx = pickle.load(f)
    with open(os.path.join(DATA_DIR, 'biz2idx.pkl'), 'rb') as f: biz2idx = pickle.load(f)
    n_users, n_biz = len(usr2idx), len(biz2idx)
    print(f'  Users: {n_users:,} | Businesses: {n_biz:,}')

    df_train = pd.read_csv(os.path.join(DATA_DIR, 'pa_reviews_train.csv'))
    df_val   = pd.read_csv(os.path.join(DATA_DIR, 'pa_reviews_val.csv'))
    df_test  = pd.read_csv(os.path.join(DATA_DIR, 'pa_reviews_test.csv'))

    print(f'  Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}')

    # User features (no friends column)
    df_users = pd.read_csv(os.path.join(DATA_DIR, 'pa_users_feat.csv'))
    feat_cols_u = [c for c in df_users.columns if c != 'user_id']
    user_feat = build_feature_matrix(df_users, 'user_id', usr2idx, feat_cols_u)
    Su_imp = np.load(os.path.join(DATA_DIR, 'Su_imp.npy'))
    S_u = np.concatenate([user_feat, Su_imp], axis=1).astype(np.float32)
    print(f'  User features: {S_u.shape[1]} ({len(feat_cols_u)} explicit + {Su_imp.shape[1]} implicit)')

    # Business features
    df_biz = pd.read_csv(os.path.join(DATA_DIR, 'pa_biz_features.csv'))
    feat_cols_b = [c for c in df_biz.columns if c != 'business_id']
    biz_feat = build_feature_matrix(df_biz, 'business_id', biz2idx, feat_cols_b)
    Sb_imp = np.load(os.path.join(DATA_DIR, 'Sb_imp.npy'))
    S_b = np.concatenate([biz_feat, Sb_imp], axis=1).astype(np.float32)
    print(f'  Biz features: {S_b.shape[1]} ({len(feat_cols_b)} explicit + {Sb_imp.shape[1]} implicit)')

    # Graphs (3 biz graphs loaded separately for omega)
    def load_edge(fname):
        G = sp.load_npz(os.path.join(DATA_DIR, fname)).tocoo()
        return torch.tensor(np.vstack([G.row, G.col]), dtype=torch.long)

    edge_u   = load_edge('user_graph.npz')
    edge_geo = load_edge('biz_graph_geo.npz')
    edge_cov = load_edge('biz_graph_covisit.npz')
    edge_cat = load_edge('biz_graph_cat.npz')
    print(f'  Edges: user={edge_u.shape[1]:,} geo={edge_geo.shape[1]:,} cov={edge_cov.shape[1]:,} cat={edge_cat.shape[1]:,}')

    return (usr2idx, biz2idx, S_u, S_b,
            edge_u, edge_geo, edge_cov, edge_cat,
            df_train, df_val, df_test, n_users, n_biz)

(usr2idx, biz2idx, S_u_np, S_b_np,
 edge_u, edge_geo, edge_cov, edge_cat,
 df_train, df_val, df_test, n_users, n_biz) = load_data()

Loading data...
  Users: 320,212 | Businesses: 31,663
  Train: 820,496 | Val: 179,662 | Test: 182,968
  User features: 65 (33 explicit + 32 implicit)
  Biz features: 1428 (1396 explicit + 32 implicit)
  Edges: user=1,744,826 geo=402,063 cov=492,463 cat=585,718


## Prepare Tensors

In [21]:
S_u = torch.tensor(S_u_np, device=DEVICE)
S_b = torch.tensor(S_b_np, device=DEVICE)
edge_u   = edge_u.to(DEVICE)
edge_geo = edge_geo.to(DEVICE)
edge_cov = edge_cov.to(DEVICE)
edge_cat = edge_cat.to(DEVICE)
edge_indices_b = [edge_geo, edge_cov, edge_cat]

# Build rating index tensors directly on GPU (no DataLoader needed)
def build_rating_tensors(df, usr2idx, biz2idx):
    valid = df[df['user_id'].isin(usr2idx) & df['business_id'].isin(biz2idx)]
    u = torch.tensor([usr2idx[r] for r in valid['user_id']], dtype=torch.long, device=DEVICE)
    b = torch.tensor([biz2idx[r] for r in valid['business_id']], dtype=torch.long, device=DEVICE)
    y = torch.tensor(valid['stars'].values, dtype=torch.float32, device=DEVICE)
    return u, b, y

train_u, train_b, train_y = build_rating_tensors(df_train, usr2idx, biz2idx)
val_u, val_b, val_y       = build_rating_tensors(df_val, usr2idx, biz2idx)
test_u, test_b, test_y    = build_rating_tensors(df_test, usr2idx, biz2idx)
print(f'Ratings: train={len(train_y):,} val={len(val_y):,} test={len(test_y):,}')

# Free DataFrames
del df_train, df_val, df_test, S_u_np, S_b_np
torch.cuda.empty_cache()

Ratings: train=820,496 val=179,662 test=182,968


## Config

In [22]:
import os                                                 
for f in ['checkpoint.pt', 'best_model.pt']:                                     
  p = os.path.join(WORK_DIR, f)                                              
  if os.path.exists(p): os.remove(p)                                           
print('Old checkpoints cleared.')    

Old checkpoints cleared.


In [23]:
import json
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

BEST_PATH = os.path.join(WORK_DIR, 'best_model.pt')
TRIALS_PATH = os.path.join(WORK_DIR, 'hyperopt_trials.json')

# ── Search space (paper footnote 4 & 6)────────────────────────────────────────
SEARCH_SPACE = {
    'theta1': hp.choice('theta1', [0.001, 0.01, 0.1]),
    'theta2': hp.choice('theta2', [0.001, 0.01, 0.1]),
    'kf':     hp.choice('kf',     [32, 64, 128]),
    'd0':     hp.choice('d0',     [32, 64]),
    'd1':     hp.choice('d1',     [64, 128]),
    'lr':     hp.choice('lr',     [0.001, 0.005]),
    'actv1':  hp.choice('actv1',  ['elu', 'relu', 'tanh']),
}

# ── Training epochs per trial (fewer for speed during search)────────────────────────
SEARCH_EPOCHS   = 30    # epochs per search trial
SEARCH_PATIENCE = 8     # early stopping patience
MAX_EVALS       = 50    # max number of search trials

best_search_rmse = float('inf')
trial_results = []

def objective(params):
    global best_search_rmse

    config = {
        'd0_u': params['d0'], 'd0_b': params['d0'],
        'd1_u': params['d1'], 'd1_b': params['d1'],
        'kf':      params['kf'],
        'actv1':   params['actv1'],
        'actv2':   'relu',
        'lr':      params['lr'],
        'theta1':  params['theta1'],
        'theta2':  params['theta2'],
        'weight_decay': 1e-5,
        'epochs':  SEARCH_EPOCHS,
        'patience': SEARCH_PATIENCE,
    }

    print(f'\n--- Trial | theta1={config["theta1"]} theta2={config["theta2"]} '
          f'kf={config["kf"]} d0={config["d0_u"]} d1={config["d1_u"]} '
          f'lr={config["lr"]} actv1={config["actv1"]} ---')

    # Build model
    trial_model = MGGAT(
        su=S_u.shape[1], sb=S_b.shape[1],
        n_users=n_users, n_biz=n_biz,
        n_biz_graphs=3,
        d0_u=config['d0_u'], d0_b=config['d0_b'],
        d1_u=config['d1_u'], d1_b=config['d1_b'],
        kf=config['kf'],
        actv1=config['actv1'],
        actv2=config['actv2'],
    ).to(DEVICE)

    opt = Adam(trial_model.parameters(),
               lr=config['lr'], weight_decay=config['weight_decay'])
    sched = ReduceLROnPlateau(opt, patience=3, factor=0.5)
    scaler = torch.cuda.amp.GradScaler()

    best_val = float('inf')
    no_improve = 0

    for epoch in range(1, config['epochs'] + 1):
        trial_model.train()
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            pred, _, _ = trial_model(S_u, S_b, edge_u, edge_indices_b, train_u, train_b)
            mse = F.mse_loss(pred, train_y)
            lreg_u = graph_laplacian_reg(trial_model.H4_u.weight, edge_u,   config['theta2'])
            lreg_b = graph_laplacian_reg(trial_model.H4_b.weight, edge_geo, config['theta2']) \
                   + graph_laplacian_reg(trial_model.H4_b.weight, edge_cov, config['theta2']) \
                   + graph_laplacian_reg(trial_model.H4_b.weight, edge_cat, config['theta2'])
            loss = mse + config['theta1'] * (lreg_u + lreg_b)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(trial_model.parameters(), 5.0)
        scaler.step(opt)
        scaler.update()

        # val
        trial_model.eval()
        with torch.no_grad(), torch.cuda.amp.autocast():
            val_pred, _, _ = trial_model(S_u, S_b, edge_u, edge_indices_b, val_u, val_b)
            val_rmse = math.sqrt(F.mse_loss(val_pred, val_y).item())
        sched.step(val_rmse)

        if val_rmse < best_val:
            best_val = val_rmse
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= config['patience']:
            break

    print(f'    → Val RMSE: {best_val:.4f}')

    # Record results
    result = {**config, 'val_rmse': best_val}
    trial_results.append(result)
    with open(TRIALS_PATH, 'w') as f:
        json.dump(trial_results, f, indent=2)

    # Save best model
    if best_val < best_search_rmse:
        best_search_rmse = best_val
        torch.save(trial_model.state_dict(), BEST_PATH)
        print(f'    ★ New best: {best_val:.4f}')

    del trial_model
    torch.cuda.empty_cache()

    return {'loss': best_val, 'status': STATUS_OK}


# ── Run search ──────────────────────────────────────────────────────────────
print(f'Starting Hyperopt search: {MAX_EVALS} trials × {SEARCH_EPOCHS} epochs each')
print('─' * 60)

trials = Trials()
best_params = fmin(
    fn=objective,
    space=SEARCH_SPACE,
    algo=tpe.suggest,
    max_evals=MAX_EVALS,
    trials=trials,
    verbose=False,
)

print('\n' + '─' * 60)
print(f'Best val RMSE: {best_search_rmse:.4f}')
print(f'Best params: {best_params}')
print(f'Trial results saved to: {TRIALS_PATH}')

# ── Full training with best params ──────────────────────────────────────────────────
# Find the best config
best_trial = min(trial_results, key=lambda x: x['val_rmse'])
print(f'\nBest config: {best_trial}')

Starting Hyperopt search: 50 trials × 30 epochs each
────────────────────────────────────────────────────────────

--- Trial | theta1=0.1 theta2=0.001 kf=64 d0=32 d1=64 lr=0.001 actv1=tanh ---


/tmp/ipykernel_141/272791560.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_141/272791560.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_141/272791560.py:86: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


    → Val RMSE: 1.4444
    ★ New best: 1.4444

--- Trial | theta1=0.01 theta2=0.001 kf=128 d0=32 d1=64 lr=0.001 actv1=elu ---
    → Val RMSE: 1.4203
    ★ New best: 1.4203

--- Trial | theta1=0.01 theta2=0.1 kf=128 d0=64 d1=64 lr=0.005 actv1=tanh ---
    → Val RMSE: 1.3832
    ★ New best: 1.3832

--- Trial | theta1=0.001 theta2=0.001 kf=64 d0=32 d1=64 lr=0.005 actv1=tanh ---
    → Val RMSE: 1.4452

--- Trial | theta1=0.01 theta2=0.1 kf=32 d0=64 d1=64 lr=0.005 actv1=relu ---
    → Val RMSE: 1.4082

--- Trial | theta1=0.01 theta2=0.001 kf=32 d0=64 d1=128 lr=0.001 actv1=tanh ---
    → Val RMSE: 1.4163

--- Trial | theta1=0.01 theta2=0.01 kf=64 d0=32 d1=128 lr=0.001 actv1=tanh ---
    → Val RMSE: 1.4104

--- Trial | theta1=0.1 theta2=0.01 kf=32 d0=32 d1=64 lr=0.005 actv1=relu ---
    → Val RMSE: 1.4143

--- Trial | theta1=0.1 theta2=0.01 kf=32 d0=64 d1=64 lr=0.001 actv1=elu ---
    → Val RMSE: 1.4242

--- Trial | theta1=0.1 theta2=0.1 kf=128 d0=64 d1=64 lr=0.005 actv1=tanh ---
    → Val RM

In [24]:
import os
for f in ['checkpoint_final.pt', 'best_model_final.pt']:
    p = os.path.join(WORK_DIR, f)
    if os.path.exists(p):
        os.remove(p)
print('Old checkpoints cleared.')

Old checkpoints cleared.


In [25]:
# Full training with best hyperparameters (from Hyperopt search)
best_trial = min(trial_results, key=lambda x: x['val_rmse'])
BEST_CONFIG = {
    **{k: best_trial[k] for k in
       ('d0_u', 'd0_b', 'd1_u', 'd1_b', 'kf', 'actv1', 'actv2',
        'lr', 'theta1', 'theta2', 'weight_decay')},
    'epochs':  200,
    'patience': 15,
}

CKPT_PATH_FINAL = os.path.join(WORK_DIR, 'checkpoint_final.pt')
BEST_PATH_FINAL = os.path.join(WORK_DIR, 'best_model_final.pt')

# Build model
model_final = MGGAT(
    su=S_u.shape[1], sb=S_b.shape[1],
    n_users=n_users, n_biz=n_biz, n_biz_graphs=3,
    d0_u=BEST_CONFIG['d0_u'], d0_b=BEST_CONFIG['d0_b'],
    d1_u=BEST_CONFIG['d1_u'], d1_b=BEST_CONFIG['d1_b'],
    kf=BEST_CONFIG['kf'],
    actv1=BEST_CONFIG['actv1'], actv2=BEST_CONFIG['actv2'],
).to(DEVICE)

optimizer_f = Adam(model_final.parameters(), lr=BEST_CONFIG['lr'], weight_decay=BEST_CONFIG['weight_decay'])
scheduler_f = ReduceLROnPlateau(optimizer_f, patience=5, factor=0.5)
scaler_f    = torch.amp.GradScaler('cuda')

# Resume from checkpoint
start_epoch_f, best_val_f, best_epoch_f = 0, float('inf'), 0
if os.path.exists(CKPT_PATH_FINAL):
    ckpt = torch.load(CKPT_PATH_FINAL, map_location=DEVICE, weights_only=False)
    model_final.load_state_dict(ckpt['model_state'])
    optimizer_f.load_state_dict(ckpt['optim_state'])
    scheduler_f.load_state_dict(ckpt['scheduler_state'])
    scaler_f.load_state_dict(ckpt['scaler_state'])
    start_epoch_f = ckpt['epoch']
    best_val_f    = ckpt['best_val_rmse']
    best_epoch_f  = ckpt['best_epoch']
    print(f'Resumed from epoch {start_epoch_f}, best val: {best_val_f:.4f}')
else:
    print('Fresh start.')

print(f'Parameters: {sum(p.numel() for p in model_final.parameters()):,}')
print(f'theta1={BEST_CONFIG["theta1"]} theta2={BEST_CONFIG["theta2"]} kf={BEST_CONFIG["kf"]} lr={BEST_CONFIG["lr"]}')
print('─' * 60)

def evaluate_final(u_idx, b_idx, y):
    model_final.eval()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        pred, _, _ = model_final(S_u, S_b, edge_u, edge_indices_b, u_idx, b_idx)
        return math.sqrt(F.mse_loss(pred, y).item())

no_improve_f = 0
for epoch in range(start_epoch_f + 1, BEST_CONFIG['epochs'] + 1):
    model_final.train()
    optimizer_f.zero_grad()

    with torch.amp.autocast('cuda'):
        pred, _, _ = model_final(S_u, S_b, edge_u, edge_indices_b, train_u, train_b)
        mse = F.mse_loss(pred, train_y)
        lreg_u = graph_laplacian_reg(model_final.H4_u.weight, edge_u,   BEST_CONFIG['theta2'])
        lreg_b = graph_laplacian_reg(model_final.H4_b.weight, edge_geo, BEST_CONFIG['theta2']) \
               + graph_laplacian_reg(model_final.H4_b.weight, edge_cov, BEST_CONFIG['theta2']) \
               + graph_laplacian_reg(model_final.H4_b.weight, edge_cat, BEST_CONFIG['theta2'])
        loss = mse + BEST_CONFIG['theta1'] * (lreg_u + lreg_b)

    scaler_f.scale(loss).backward()
    scaler_f.unscale_(optimizer_f)
    nn.utils.clip_grad_norm_(model_final.parameters(), max_norm=5.0)
    scaler_f.step(optimizer_f)
    scaler_f.update()

    train_rmse = math.sqrt(mse.item())
    val_rmse   = evaluate_final(val_u, val_b, val_y)
    scheduler_f.step(val_rmse)

    flag = ''
    if val_rmse < best_val_f:
        best_val_f   = val_rmse
        best_epoch_f = epoch
        no_improve_f = 0
        torch.save(model_final.state_dict(), BEST_PATH_FINAL)
        flag = ' <- best'
    else:
        no_improve_f += 1

    lr_now = optimizer_f.param_groups[0]['lr']
    print(f'Epoch {epoch:3d} | Train: {train_rmse:.4f} | Val: {val_rmse:.4f} | lr={lr_now:.1e}{flag}')

    # Save checkpoint
    torch.save({
        'epoch':           epoch,
        'model_state':     model_final.state_dict(),
        'optim_state':     optimizer_f.state_dict(),
        'scheduler_state': scheduler_f.state_dict(),
        'scaler_state':    scaler_f.state_dict(),
        'best_val_rmse':   best_val_f,
        'best_epoch':      best_epoch_f,
    }, CKPT_PATH_FINAL)

    if no_improve_f >= BEST_CONFIG['patience']:
        print(f'\nEarly stopping at epoch {epoch}')
        break

# Test set evaluation
print('\n' + '─' * 60)
model_final.load_state_dict(torch.load(BEST_PATH_FINAL, map_location=DEVICE, weights_only=False))
test_rmse = evaluate_final(test_u, test_b, test_y)
print(f'Best val RMSE:  {best_val_f:.4f} (epoch {best_epoch_f})')
print(f'Test RMSE:      {test_rmse:.4f}  (paper target: ~1.249)')

Fresh start.
Parameters: 45,607,943
theta1=0.01 theta2=0.1 kf=128 lr=0.005
────────────────────────────────────────────────────────────
Epoch   1 | Train: 1.4987 | Val: 1.7912 | lr=5.0e-03 <- best
Epoch   2 | Train: 1.7422 | Val: 1.4810 | lr=5.0e-03 <- best
Epoch   3 | Train: 1.3691 | Val: 1.5574 | lr=5.0e-03
Epoch   4 | Train: 1.4357 | Val: 1.4799 | lr=5.0e-03 <- best
Epoch   5 | Train: 1.3737 | Val: 1.4671 | lr=5.0e-03 <- best
Epoch   6 | Train: 1.3514 | Val: 1.4789 | lr=5.0e-03
Epoch   7 | Train: 1.3523 | Val: 1.4583 | lr=5.0e-03 <- best
Epoch   8 | Train: 1.3356 | Val: 1.4531 | lr=5.0e-03 <- best
Epoch   9 | Train: 1.3344 | Val: 1.4607 | lr=5.0e-03
Epoch  10 | Train: 1.3282 | Val: 1.4428 | lr=5.0e-03 <- best
Epoch  11 | Train: 1.3116 | Val: 1.4433 | lr=5.0e-03
Epoch  12 | Train: 1.3184 | Val: 1.4384 | lr=5.0e-03 <- best
Epoch  13 | Train: 1.3006 | Val: 1.4321 | lr=5.0e-03 <- best
Epoch  14 | Train: 1.2909 | Val: 1.4263 | lr=5.0e-03 <- best
Epoch  15 | Train: 1.2928 | Val: 1.4201 | 

In [26]:
# ══════════════════════════════════════════════════════════════════════════
# Ablation experiments
# Paper reference: Section 5.3, Table 2 (PA subset)
# Run two ablation variants using BEST_CONFIG
# ══════════════════════════════════════════════════════════════════════════

ABLATION_CONFIG = dict(BEST_CONFIG)  # derived from Hyperopt best trial

def run_ablation(name, su, sb, n_users, n_biz, config,
                 ablate_nig=False, ablate_fr=False):
    """
    name:       experiment name
    ablate_nig: True → remove NIG, use uniform weights [Section 5.3, Table 2 "NIG Removed"]
    ablate_fr:  True → remove FR, make Layer 1 nonlinear [Section 5.3, Table 2 "FR Removed"]
    """
    print(f'\n{"="*60}')
    print(f'Ablation: {name}')
    print(f'  ablate_nig={ablate_nig} | ablate_fr={ablate_fr}')
    print(f'{"="*60}')

    # ── Ablation variant of MGGAT ──────────────────────────────────────────────────────
    class MGGATAblation(nn.Module):
        def __init__(self):
            super().__init__()
            self.kf    = config['kf']
            self.r_min = 1.0
            self.r_max = 5.0
            self.d0_u  = config['d0_u']
            self.d0_b  = config['d0_b']

            # Layer 1
            self.W1_u = nn.Linear(su, config['d0_u'], bias=False)
            self.W1_b = nn.Linear(sb, config['d0_b'], bias=False)

            # Layer 2 attention (unused when ablate_nig=True)
            self.a_u  = nn.Parameter(torch.empty(2 * config['d0_u']))
            self.a_b  = nn.Parameter(torch.empty(2 * config['d0_b']))
            nn.init.xavier_uniform_(self.a_u.unsqueeze(0))
            nn.init.xavier_uniform_(self.a_b.unsqueeze(0))
            self.omega     = nn.Parameter(torch.ones(3) / 3)
            self.leaky_relu = nn.LeakyReLU(0.2)

            # Layer 4
            self.W2_u  = nn.Linear(config['d0_u'], config['d1_u'], bias=False)
            self.W2_us = nn.Linear(su, config['d1_u'], bias=False)
            self.b1_u  = nn.Parameter(torch.zeros(config['d1_u']))
            self.W2_b  = nn.Linear(config['d0_b'], config['d1_b'], bias=False)
            self.W2_bs = nn.Linear(sb, config['d1_b'], bias=False)
            self.b1_b  = nn.Parameter(torch.zeros(config['d1_b']))

            # Layer 5
            self.W3_u = nn.Linear(config['d1_u'], config['kf'], bias=False)
            self.W3_b = nn.Linear(config['d1_b'], config['kf'], bias=False)
            self.H4_u = nn.Embedding(n_users, config['kf'])
            self.H4_b = nn.Embedding(n_biz,   config['kf'])
            nn.init.normal_(self.H4_u.weight, std=0.01)
            nn.init.normal_(self.H4_b.weight, std=0.01)

            # Biases
            self.bias_u      = nn.Embedding(n_users, 1)
            self.bias_b      = nn.Embedding(n_biz,   1)
            self.bias_global = nn.Parameter(torch.zeros(1))
            nn.init.zeros_(self.bias_u.weight)
            nn.init.zeros_(self.bias_b.weight)

            acts = {'elu': nn.ELU(), 'relu': nn.ReLU(),
                    'tanh': nn.Tanh(), 'sigmoid': nn.Sigmoid()}
            self.actv1 = acts[config['actv1']]
            self.actv2 = acts[config['actv2']]

            # FR ablation: extra nonlinear activation
            if ablate_fr:
                self.fr_act = nn.Sigmoid()   # Appendix E, Eq.13

        def _softmax_by_dst(self, e, dst, N):
            e_max = torch.full((N,), float('-inf'), device=e.device)
            e_max.scatter_reduce_(0, dst, e, reduce='amax', include_self=True)
            e_exp = torch.exp(e - e_max[dst])
            e_sum = torch.zeros(N, device=e.device).scatter_add_(0, dst, e_exp)
            return e_exp / (e_sum[dst] + 1e-16)

        def _nig_user(self, H1_u, edge_index_u):
            src, dst = edge_index_u[0], edge_index_u[1]
            N = H1_u.shape[0]

            if ablate_nig:
                # Remove NIG: uniform weights [Table 2 "NIG Removed"]
                # α(k→i) = 1 / |N_i|
                deg = torch.zeros(N, device=H1_u.device)
                deg.scatter_add_(0, dst, torch.ones(len(dst), device=H1_u.device))
                deg = torch.clamp(deg, min=1)
                return 1.0 / deg[dst]
            else:
                a_dst = self.a_u[:self.d0_u]
                a_src = self.a_u[self.d0_u:]
                e = self.leaky_relu(
                    (H1_u[dst] * a_dst).sum(1) + (H1_u[src] * a_src).sum(1)
                )
                return self._softmax_by_dst(e, dst, N)

        def _nig_biz(self, H1_b, edge_indices_b):
            omega = torch.softmax(self.omega, dim=0)
            N = H1_b.shape[0]
            all_src, all_dst, all_e = [], [], []

            for g, ei in enumerate(edge_indices_b):
                s, d = ei[0], ei[1]
                if ablate_nig:
                    # Uniform weight placeholder (handled below)
                    all_src.append(s)
                    all_dst.append(d)
                    all_e.append(torch.ones(len(s), device=H1_b.device))
                else:
                    a_dst = self.a_b[:self.d0_b]
                    a_src = self.a_b[self.d0_b:]
                    e_g = (H1_b[d] * a_dst).sum(1) + (H1_b[s] * a_src).sum(1)
                    all_src.append(s)
                    all_dst.append(d)
                    all_e.append(omega[g] * e_g)

            all_src = torch.cat(all_src)
            all_dst = torch.cat(all_dst)
            all_e   = torch.cat(all_e)

            if ablate_nig:
                deg = torch.zeros(N, device=H1_b.device)
                deg.scatter_add_(0, all_dst, torch.ones(len(all_dst), device=H1_b.device))
                deg = torch.clamp(deg, min=1)
                alpha = 1.0 / deg[all_dst]
            else:
                all_e = self.leaky_relu(all_e)
                alpha = self._softmax_by_dst(all_e, all_dst, N)

            return all_src, all_dst, alpha

        def _aggregate(self, H1, alpha, src, dst, N):
            weighted = H1[src] * alpha.unsqueeze(1)
            H2 = torch.zeros(N, H1.shape[1], device=H1.device)
            H2.scatter_add_(0, dst.unsqueeze(1).expand_as(weighted), weighted)
            return H2

        def forward(self, S_u, S_b, edge_index_u, edge_indices_b, user_idx, biz_idx):
            n_u, n_b = S_u.shape[0], S_b.shape[0]

            # Layer 1
            H1_u = self.W1_u(S_u)
            H1_b = self.W1_b(S_b)

            if ablate_fr:
                # Remove FR: add nonlinearity [Appendix E, Eq.13]
                H1_u = self.fr_act(H1_u)
                H1_b = self.fr_act(H1_b)

            # Layer 2: NIG
            alpha_u = self._nig_user(H1_u, edge_index_u)
            src_b, dst_b, alpha_b = self._nig_biz(H1_b, edge_indices_b)

            # Layer 3: aggregation
            src_u, dst_u = edge_index_u[0], edge_index_u[1]
            H2_u = self._aggregate(H1_u, alpha_u, src_u, dst_u, n_u)
            H2_b = self._aggregate(H1_b, alpha_b, src_b, dst_b, n_b)

            # Layer 4
            H3_u = self.actv1(self.W2_u(H2_u) + self.W2_us(S_u) + self.b1_u)
            H3_b = self.actv1(self.W2_b(H2_b) + self.W2_bs(S_b) + self.b1_b)

            # Layer 5
            U_all = self.actv2(self.W3_u(H3_u)) + self.H4_u.weight
            B_all = self.actv2(self.W3_b(H3_b)) + self.H4_b.weight

            U_q, B_q = U_all[user_idx], B_all[biz_idx]
            logit = (U_q * B_q).sum(1) \
                  + self.bias_u(user_idx).squeeze(1) \
                  + self.bias_b(biz_idx).squeeze(1) \
                  + self.bias_global
            pred = (self.r_max - self.r_min) * torch.sigmoid(logit) + self.r_min
            return pred, U_all, B_all

    # ── Training ──────────────────────────────────────────────────────────────
    model_abl = MGGATAblation().to(DEVICE)
    opt_abl   = Adam(model_abl.parameters(),
                     lr=config['lr'], weight_decay=config['weight_decay'])
    sched_abl = ReduceLROnPlateau(opt_abl, patience=5, factor=0.5)
    scaler_abl = torch.amp.GradScaler('cuda')

    best_val, best_epoch_abl, no_imp = float('inf'), 0, 0
    best_state = None

    for epoch in range(1, config['epochs'] + 1):
        model_abl.train()
        opt_abl.zero_grad()
        with torch.amp.autocast('cuda'):
            pred, _, _ = model_abl(S_u, S_b, edge_u, edge_indices_b, train_u, train_b)
            mse = F.mse_loss(pred, train_y)
            lreg_u = graph_laplacian_reg(model_abl.H4_u.weight, edge_u,   config['theta2'])
            lreg_b = graph_laplacian_reg(model_abl.H4_b.weight, edge_geo, config['theta2']) \
                   + graph_laplacian_reg(model_abl.H4_b.weight, edge_cov, config['theta2']) \
                   + graph_laplacian_reg(model_abl.H4_b.weight, edge_cat, config['theta2'])
            loss = mse + config['theta1'] * (lreg_u + lreg_b)
        scaler_abl.scale(loss).backward()
        scaler_abl.unscale_(opt_abl)
        nn.utils.clip_grad_norm_(model_abl.parameters(), 5.0)
        scaler_abl.step(opt_abl)
        scaler_abl.update()

        model_abl.eval()
        with torch.no_grad(), torch.amp.autocast('cuda'):
            vp, _, _ = model_abl(S_u, S_b, edge_u, edge_indices_b, val_u, val_b)
            val_rmse = math.sqrt(F.mse_loss(vp, val_y).item())
        sched_abl.step(val_rmse)

        if val_rmse < best_val:
            best_val       = val_rmse
            best_epoch_abl = epoch
            no_imp         = 0
            best_state     = {k: v.clone() for k, v in model_abl.state_dict().items()}
        else:
            no_imp += 1

        lr_now = opt_abl.param_groups[0]['lr']
        flag = ' <- best' if no_imp == 0 else ''
        print(f'Epoch {epoch:3d} | Train: {math.sqrt(mse.item()):.4f} | Val: {val_rmse:.4f} | lr={lr_now:.1e}{flag}')

        if no_imp >= config['patience']:
            print(f'Early stopping at epoch {epoch}')
            break

    # Test set
    model_abl.load_state_dict(best_state)
    model_abl.eval()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        tp, _, _ = model_abl(S_u, S_b, edge_u, edge_indices_b, test_u, test_b)
        test_rmse = math.sqrt(F.mse_loss(tp, test_y).item())

    print(f'\n{name}')
    print(f'  Best val RMSE:  {best_val:.4f} (epoch {best_epoch_abl})')
    print(f'  Test RMSE:      {test_rmse:.4f}')

    del model_abl
    torch.cuda.empty_cache()
    return best_val, test_rmse


# ── Run two ablation experiments ──────────────────────────────────────────────────────
su = S_u.shape[1]
sb = S_b.shape[1]

val_nig, test_nig = run_ablation(
    'Ablation 1: NIG Removed (uniform weights)',
    su, sb, n_users, n_biz, ABLATION_CONFIG,
    ablate_nig=True, ablate_fr=False
)

val_fr, test_fr = run_ablation(
    'Ablation 2: FR Removed (nonlinear Layer 1)',
    su, sb, n_users, n_biz, ABLATION_CONFIG,
    ablate_nig=False, ablate_fr=True
)

# ── Summary ──────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('Ablation Study Summary')
print('='*60)
print(f'{"Configuration":<40} {"Val RMSE":>10} {"Test RMSE":>10}')
print('-'*60)
print(f'{"Full MG-GAT (Final)":<40} {best_val_f:>10.4f} {test_rmse:>10.4f}')
print(f'{"NIG Removed (uniform weights)":<40} {val_nig:>10.4f} {test_nig:>10.4f}')
print(f'{"FR Removed (nonlinear Layer 1)":<40} {val_fr:>10.4f} {test_fr:>10.4f}')
print('-'*60)
print(f'Paper: Full MG-GAT                         {"—":>10} {"1.249":>10}')
print(f'Paper: NIG Removed                         {"—":>10} {"1.303":>10}')
print(f'Paper: FR Removed                          {"—":>10} {"1.305":>10}')


Ablation: Ablation 1: NIG Removed (uniform weights)
  ablate_nig=True | ablate_fr=False
Epoch   1 | Train: 1.5014 | Val: 1.7752 | lr=5.0e-03 <- best
Epoch   2 | Train: 1.7249 | Val: 1.4817 | lr=5.0e-03 <- best
Epoch   3 | Train: 1.3779 | Val: 1.5803 | lr=5.0e-03
Epoch   4 | Train: 1.4581 | Val: 1.4801 | lr=5.0e-03 <- best
Epoch   5 | Train: 1.3630 | Val: 1.5857 | lr=5.0e-03
Epoch   6 | Train: 1.5079 | Val: 1.4680 | lr=5.0e-03 <- best
Epoch   7 | Train: 1.3480 | Val: 1.5121 | lr=5.0e-03
Epoch   8 | Train: 1.3847 | Val: 1.4812 | lr=5.0e-03
Epoch   9 | Train: 1.3549 | Val: 1.4790 | lr=5.0e-03
Epoch  10 | Train: 1.3666 | Val: 1.4550 | lr=5.0e-03 <- best
Epoch  11 | Train: 1.3334 | Val: 1.4616 | lr=5.0e-03
Epoch  12 | Train: 1.3297 | Val: 1.4515 | lr=5.0e-03 <- best
Epoch  13 | Train: 1.3192 | Val: 1.4405 | lr=5.0e-03 <- best
Epoch  14 | Train: 1.3183 | Val: 1.4318 | lr=5.0e-03 <- best
Epoch  15 | Train: 1.3006 | Val: 1.4376 | lr=5.0e-03
Epoch  16 | Train: 1.2996 | Val: 1.4217 | lr=5.0e-03

## Additional Ablation Experiments (Table 2)

Three more ablation variants from the paper [Section 5.3, Table 2]:

1. **Uniform Graph Weighting** — fix ω_g = 1/3 for all business graphs (not learned)
2. **No Auxiliary Information** — remove side features S_u, S_b; use learnable embeddings instead
3. **No Networks or Auxiliary Information** — remove both graphs and side features; pure matrix factorization

In [27]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ══════════════════════════════════════════════════════════════════════════
# Ablation 3: Uniform Graph Weighting
# Paper Table 2: omega_g fixed at 1/3 for all 3 business graphs
# ══════════════════════════════════════════════════════════════════════════

class MGGATUniformOmega(nn.Module):
    """Same as full MG-GAT but omega weights are fixed (not learned)."""
    def __init__(self, su, sb, n_users, n_biz, config):
        super().__init__()
        self.kf = config['kf']
        self.r_min, self.r_max = 1.0, 5.0
        self.d0_u, self.d0_b = config['d0_u'], config['d0_b']

        self.W1_u = nn.Linear(su, config['d0_u'], bias=False)
        self.W1_b = nn.Linear(sb, config['d0_b'], bias=False)

        self.a_u = nn.Parameter(torch.empty(2 * config['d0_u']))
        self.a_b = nn.Parameter(torch.empty(2 * config['d0_b']))
        nn.init.xavier_uniform_(self.a_u.unsqueeze(0))
        nn.init.xavier_uniform_(self.a_b.unsqueeze(0))
        # Fixed uniform omega — NOT a learnable parameter
        self.register_buffer('omega_fixed', torch.ones(3) / 3)
        self.leaky_relu = nn.LeakyReLU(0.2)

        self.W2_u  = nn.Linear(config['d0_u'], config['d1_u'], bias=False)
        self.W2_us = nn.Linear(su, config['d1_u'], bias=False)
        self.b1_u  = nn.Parameter(torch.zeros(config['d1_u']))
        self.W2_b  = nn.Linear(config['d0_b'], config['d1_b'], bias=False)
        self.W2_bs = nn.Linear(sb, config['d1_b'], bias=False)
        self.b1_b  = nn.Parameter(torch.zeros(config['d1_b']))

        self.W3_u = nn.Linear(config['d1_u'], config['kf'], bias=False)
        self.W3_b = nn.Linear(config['d1_b'], config['kf'], bias=False)
        self.H4_u = nn.Embedding(n_users, config['kf'])
        self.H4_b = nn.Embedding(n_biz, config['kf'])
        nn.init.normal_(self.H4_u.weight, std=0.01)
        nn.init.normal_(self.H4_b.weight, std=0.01)

        self.bias_u = nn.Embedding(n_users, 1)
        self.bias_b = nn.Embedding(n_biz, 1)
        self.bias_global = nn.Parameter(torch.zeros(1))
        nn.init.zeros_(self.bias_u.weight)
        nn.init.zeros_(self.bias_b.weight)

        acts = {'elu': nn.ELU(), 'relu': nn.ReLU(), 'tanh': nn.Tanh(), 'sigmoid': nn.Sigmoid()}
        self.actv1 = acts[config['actv1']]
        self.actv2 = acts[config['actv2']]

    def _softmax_by_dst(self, e, dst, N):
        e_max = torch.full((N,), float('-inf'), device=e.device)
        e_max.scatter_reduce_(0, dst, e, reduce='amax', include_self=True)
        e_exp = torch.exp(e - e_max[dst])
        e_sum = torch.zeros(N, device=e.device).scatter_add_(0, dst, e_exp)
        return e_exp / (e_sum[dst] + 1e-16)

    def forward(self, S_u, S_b, edge_index_u, edge_indices_b, user_idx, biz_idx):
        n_u, n_b = S_u.shape[0], S_b.shape[0]
        H1_u = self.W1_u(S_u)
        H1_b = self.W1_b(S_b)

        # User NIG (same as full model)
        src_u, dst_u = edge_index_u[0], edge_index_u[1]
        a_dst_u, a_src_u = self.a_u[:self.d0_u], self.a_u[self.d0_u:]
        e_u = self.leaky_relu((H1_u[dst_u] * a_dst_u).sum(1) + (H1_u[src_u] * a_src_u).sum(1))
        alpha_u = self._softmax_by_dst(e_u, dst_u, n_u)

        # Business NIG with FIXED omega (uniform 1/3)
        a_dst_b, a_src_b = self.a_b[:self.d0_b], self.a_b[self.d0_b:]
        all_src, all_dst, all_e = [], [], []
        for g, ei in enumerate(edge_indices_b):
            s, d = ei[0], ei[1]
            e_g = (H1_b[d] * a_dst_b).sum(1) + (H1_b[s] * a_src_b).sum(1)
            all_src.append(s); all_dst.append(d)
            all_e.append(self.omega_fixed[g] * e_g)  # fixed 1/3
        all_src = torch.cat(all_src); all_dst = torch.cat(all_dst)
        all_e = self.leaky_relu(torch.cat(all_e))
        alpha_b = self._softmax_by_dst(all_e, all_dst, n_b)

        # Aggregate
        wu = H1_u[src_u] * alpha_u.unsqueeze(1)
        H2_u = torch.zeros(n_u, H1_u.shape[1], device=H1_u.device)
        H2_u.scatter_add_(0, dst_u.unsqueeze(1).expand_as(wu), wu)

        wb = H1_b[all_src] * alpha_b.unsqueeze(1)
        H2_b = torch.zeros(n_b, H1_b.shape[1], device=H1_b.device)
        H2_b.scatter_add_(0, all_dst.unsqueeze(1).expand_as(wb), wb)

        H3_u = self.actv1(self.W2_u(H2_u) + self.W2_us(S_u) + self.b1_u)
        H3_b = self.actv1(self.W2_b(H2_b) + self.W2_bs(S_b) + self.b1_b)

        U_all = self.actv2(self.W3_u(H3_u)) + self.H4_u.weight
        B_all = self.actv2(self.W3_b(H3_b)) + self.H4_b.weight

        U_q, B_q = U_all[user_idx], B_all[biz_idx]
        logit = (U_q * B_q).sum(1) + self.bias_u(user_idx).squeeze(1) \
              + self.bias_b(biz_idx).squeeze(1) + self.bias_global
        pred = (self.r_max - self.r_min) * torch.sigmoid(logit) + self.r_min
        return pred, U_all, B_all


# ── Train ─────────────────────────────────────────────────────────────────
print('Ablation 3: Uniform Graph Weighting')
print('=' * 60)

model_uw = MGGATUniformOmega(S_u.shape[1], S_b.shape[1], n_users, n_biz, ABLATION_CONFIG).to(DEVICE)
opt_uw = Adam(model_uw.parameters(), lr=ABLATION_CONFIG['lr'], weight_decay=ABLATION_CONFIG['weight_decay'])
sched_uw = ReduceLROnPlateau(opt_uw, patience=5, factor=0.5)
scaler_uw = torch.amp.GradScaler('cuda')

best_val_uw, best_epoch_uw, no_imp_uw = float('inf'), 0, 0
best_state_uw = None

for epoch in range(1, ABLATION_CONFIG['epochs'] + 1):
    model_uw.train()
    opt_uw.zero_grad()
    with torch.amp.autocast('cuda'):
        pred, _, _ = model_uw(S_u, S_b, edge_u, edge_indices_b, train_u, train_b)
        mse = F.mse_loss(pred, train_y)
        lreg_u = graph_laplacian_reg(model_uw.H4_u.weight, edge_u, ABLATION_CONFIG['theta2'])
        lreg_b = graph_laplacian_reg(model_uw.H4_b.weight, edge_geo, ABLATION_CONFIG['theta2']) \
               + graph_laplacian_reg(model_uw.H4_b.weight, edge_cov, ABLATION_CONFIG['theta2']) \
               + graph_laplacian_reg(model_uw.H4_b.weight, edge_cat, ABLATION_CONFIG['theta2'])
        loss = mse + ABLATION_CONFIG['theta1'] * (lreg_u + lreg_b)
    scaler_uw.scale(loss).backward()
    scaler_uw.unscale_(opt_uw)
    nn.utils.clip_grad_norm_(model_uw.parameters(), 5.0)
    scaler_uw.step(opt_uw)
    scaler_uw.update()

    model_uw.eval()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        vp, _, _ = model_uw(S_u, S_b, edge_u, edge_indices_b, val_u, val_b)
        val_rmse = math.sqrt(F.mse_loss(vp, val_y).item())
    sched_uw.step(val_rmse)

    flag = ''
    if val_rmse < best_val_uw:
        best_val_uw = val_rmse; best_epoch_uw = epoch; no_imp_uw = 0
        best_state_uw = {k: v.clone() for k, v in model_uw.state_dict().items()}
        flag = ' <- best'
    else:
        no_imp_uw += 1

    lr_now = opt_uw.param_groups[0]['lr']
    print(f'Epoch {epoch:3d} | Train: {math.sqrt(mse.item()):.4f} | Val: {val_rmse:.4f} | lr={lr_now:.1e}{flag}')
    if no_imp_uw >= ABLATION_CONFIG['patience']:
        print(f'Early stopping at epoch {epoch}'); break

model_uw.load_state_dict(best_state_uw)
model_uw.eval()
with torch.no_grad(), torch.amp.autocast('cuda'):
    tp, _, _ = model_uw(S_u, S_b, edge_u, edge_indices_b, test_u, test_b)
    test_rmse_uw = math.sqrt(F.mse_loss(tp, test_y).item())
print(f'\nUniform Graph Weighting')
print(f'  Best val RMSE:  {best_val_uw:.4f} (epoch {best_epoch_uw})')
print(f'  Test RMSE:      {test_rmse_uw:.4f}')
del model_uw; torch.cuda.empty_cache()

Ablation 3: Uniform Graph Weighting
Epoch   1 | Train: 1.5011 | Val: 1.7532 | lr=5.0e-03 <- best
Epoch   2 | Train: 1.7143 | Val: 1.4798 | lr=5.0e-03 <- best
Epoch   3 | Train: 1.3744 | Val: 1.5627 | lr=5.0e-03
Epoch   4 | Train: 1.4389 | Val: 1.4716 | lr=5.0e-03 <- best
Epoch   5 | Train: 1.3586 | Val: 1.4912 | lr=5.0e-03
Epoch   6 | Train: 1.3924 | Val: 1.4774 | lr=5.0e-03
Epoch   7 | Train: 1.3521 | Val: 1.4904 | lr=5.0e-03
Epoch   8 | Train: 1.3603 | Val: 1.4543 | lr=5.0e-03 <- best
Epoch   9 | Train: 1.3315 | Val: 1.4576 | lr=5.0e-03
Epoch  10 | Train: 1.3463 | Val: 1.4466 | lr=5.0e-03 <- best
Epoch  11 | Train: 1.3177 | Val: 1.4563 | lr=5.0e-03
Epoch  12 | Train: 1.3221 | Val: 1.4331 | lr=5.0e-03 <- best
Epoch  13 | Train: 1.3025 | Val: 1.4319 | lr=5.0e-03 <- best
Epoch  14 | Train: 1.3073 | Val: 1.4264 | lr=5.0e-03 <- best
Epoch  15 | Train: 1.2900 | Val: 1.4278 | lr=5.0e-03
Epoch  16 | Train: 1.2877 | Val: 1.4154 | lr=5.0e-03 <- best
Epoch  17 | Train: 1.2811 | Val: 1.4103 | lr

In [28]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ══════════════════════════════════════════════════════════════════════════
# Ablation 4: No Auxiliary Information
# Paper Table 2: remove S_u, S_b; use learnable embeddings as input
# Graph structure is still used for attention and regularization
# ══════════════════════════════════════════════════════════════════════════

class MGGATNoAux(nn.Module):
    """MG-GAT without side features S_u/S_b. Uses learnable embeddings instead."""
    def __init__(self, n_users, n_biz, config):
        super().__init__()
        self.kf = config['kf']
        self.r_min, self.r_max = 1.0, 5.0
        self.d0_u, self.d0_b = config['d0_u'], config['d0_b']

        # Learnable input embeddings replacing S_u and S_b
        self.emb_u = nn.Embedding(n_users, config['d0_u'])
        self.emb_b = nn.Embedding(n_biz, config['d0_b'])
        nn.init.normal_(self.emb_u.weight, std=0.01)
        nn.init.normal_(self.emb_b.weight, std=0.01)

        # No W1 needed — emb already at d0 dimension
        # No W2_us/W2_bs skip connection — no side features to skip from

        self.a_u = nn.Parameter(torch.empty(2 * config['d0_u']))
        self.a_b = nn.Parameter(torch.empty(2 * config['d0_b']))
        nn.init.xavier_uniform_(self.a_u.unsqueeze(0))
        nn.init.xavier_uniform_(self.a_b.unsqueeze(0))
        self.omega = nn.Parameter(torch.ones(3) / 3)
        self.leaky_relu = nn.LeakyReLU(0.2)

        self.W2_u = nn.Linear(config['d0_u'], config['d1_u'], bias=False)
        self.b1_u = nn.Parameter(torch.zeros(config['d1_u']))
        self.W2_b = nn.Linear(config['d0_b'], config['d1_b'], bias=False)
        self.b1_b = nn.Parameter(torch.zeros(config['d1_b']))

        self.W3_u = nn.Linear(config['d1_u'], config['kf'], bias=False)
        self.W3_b = nn.Linear(config['d1_b'], config['kf'], bias=False)
        self.H4_u = nn.Embedding(n_users, config['kf'])
        self.H4_b = nn.Embedding(n_biz, config['kf'])
        nn.init.normal_(self.H4_u.weight, std=0.01)
        nn.init.normal_(self.H4_b.weight, std=0.01)

        self.bias_u = nn.Embedding(n_users, 1)
        self.bias_b = nn.Embedding(n_biz, 1)
        self.bias_global = nn.Parameter(torch.zeros(1))
        nn.init.zeros_(self.bias_u.weight)
        nn.init.zeros_(self.bias_b.weight)

        acts = {'elu': nn.ELU(), 'relu': nn.ReLU(), 'tanh': nn.Tanh(), 'sigmoid': nn.Sigmoid()}
        self.actv1 = acts[config['actv1']]
        self.actv2 = acts[config['actv2']]

    def _softmax_by_dst(self, e, dst, N):
        e_max = torch.full((N,), float('-inf'), device=e.device)
        e_max.scatter_reduce_(0, dst, e, reduce='amax', include_self=True)
        e_exp = torch.exp(e - e_max[dst])
        e_sum = torch.zeros(N, device=e.device).scatter_add_(0, dst, e_exp)
        return e_exp / (e_sum[dst] + 1e-16)

    def forward(self, edge_index_u, edge_indices_b, user_idx, biz_idx):
        n_u, n_b = self.emb_u.weight.shape[0], self.emb_b.weight.shape[0]

        # Layer 1: use learnable embeddings directly (replaces W1 * S)
        H1_u = self.emb_u.weight
        H1_b = self.emb_b.weight

        # User NIG
        src_u, dst_u = edge_index_u[0], edge_index_u[1]
        a_dst_u, a_src_u = self.a_u[:self.d0_u], self.a_u[self.d0_u:]
        e_u = self.leaky_relu((H1_u[dst_u] * a_dst_u).sum(1) + (H1_u[src_u] * a_src_u).sum(1))
        alpha_u = self._softmax_by_dst(e_u, dst_u, n_u)

        # Business NIG
        omega = torch.softmax(self.omega, dim=0)
        a_dst_b, a_src_b = self.a_b[:self.d0_b], self.a_b[self.d0_b:]
        all_src, all_dst, all_e = [], [], []
        for g, ei in enumerate(edge_indices_b):
            s, d = ei[0], ei[1]
            e_g = (H1_b[d] * a_dst_b).sum(1) + (H1_b[s] * a_src_b).sum(1)
            all_src.append(s); all_dst.append(d)
            all_e.append(omega[g] * e_g)
        all_src = torch.cat(all_src); all_dst = torch.cat(all_dst)
        all_e = self.leaky_relu(torch.cat(all_e))
        alpha_b = self._softmax_by_dst(all_e, all_dst, n_b)

        # Aggregate
        wu = H1_u[src_u] * alpha_u.unsqueeze(1)
        H2_u = torch.zeros(n_u, H1_u.shape[1], device=H1_u.device)
        H2_u.scatter_add_(0, dst_u.unsqueeze(1).expand_as(wu), wu)
        wb = H1_b[all_src] * alpha_b.unsqueeze(1)
        H2_b = torch.zeros(n_b, H1_b.shape[1], device=H1_b.device)
        H2_b.scatter_add_(0, all_dst.unsqueeze(1).expand_as(wb), wb)

        # Layer 4 (no skip connection — no S_u/S_b)
        H3_u = self.actv1(self.W2_u(H2_u) + self.b1_u)
        H3_b = self.actv1(self.W2_b(H2_b) + self.b1_b)

        # Layer 5
        U_all = self.actv2(self.W3_u(H3_u)) + self.H4_u.weight
        B_all = self.actv2(self.W3_b(H3_b)) + self.H4_b.weight

        U_q, B_q = U_all[user_idx], B_all[biz_idx]
        logit = (U_q * B_q).sum(1) + self.bias_u(user_idx).squeeze(1) \
              + self.bias_b(biz_idx).squeeze(1) + self.bias_global
        pred = (self.r_max - self.r_min) * torch.sigmoid(logit) + self.r_min
        return pred, U_all, B_all


# ── Train ─────────────────────────────────────────────────────────────────
print('Ablation 4: No Auxiliary Information')
print('=' * 60)

model_na = MGGATNoAux(n_users, n_biz, ABLATION_CONFIG).to(DEVICE)
opt_na = Adam(model_na.parameters(), lr=ABLATION_CONFIG['lr'], weight_decay=ABLATION_CONFIG['weight_decay'])
sched_na = ReduceLROnPlateau(opt_na, patience=5, factor=0.5)
scaler_na = torch.amp.GradScaler('cuda')

best_val_na, best_epoch_na, no_imp_na = float('inf'), 0, 0
best_state_na = None

for epoch in range(1, ABLATION_CONFIG['epochs'] + 1):
    model_na.train()
    opt_na.zero_grad()
    with torch.amp.autocast('cuda'):
        pred, _, _ = model_na(edge_u, edge_indices_b, train_u, train_b)
        mse = F.mse_loss(pred, train_y)
        lreg_u = graph_laplacian_reg(model_na.H4_u.weight, edge_u, ABLATION_CONFIG['theta2'])
        lreg_b = graph_laplacian_reg(model_na.H4_b.weight, edge_geo, ABLATION_CONFIG['theta2']) \
               + graph_laplacian_reg(model_na.H4_b.weight, edge_cov, ABLATION_CONFIG['theta2']) \
               + graph_laplacian_reg(model_na.H4_b.weight, edge_cat, ABLATION_CONFIG['theta2'])
        loss = mse + ABLATION_CONFIG['theta1'] * (lreg_u + lreg_b)
    scaler_na.scale(loss).backward()
    scaler_na.unscale_(opt_na)
    nn.utils.clip_grad_norm_(model_na.parameters(), 5.0)
    scaler_na.step(opt_na)
    scaler_na.update()

    model_na.eval()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        vp, _, _ = model_na(edge_u, edge_indices_b, val_u, val_b)
        val_rmse = math.sqrt(F.mse_loss(vp, val_y).item())
    sched_na.step(val_rmse)

    flag = ''
    if val_rmse < best_val_na:
        best_val_na = val_rmse; best_epoch_na = epoch; no_imp_na = 0
        best_state_na = {k: v.clone() for k, v in model_na.state_dict().items()}
        flag = ' <- best'
    else:
        no_imp_na += 1

    lr_now = opt_na.param_groups[0]['lr']
    print(f'Epoch {epoch:3d} | Train: {math.sqrt(mse.item()):.4f} | Val: {val_rmse:.4f} | lr={lr_now:.1e}{flag}')
    if no_imp_na >= ABLATION_CONFIG['patience']:
        print(f'Early stopping at epoch {epoch}'); break

model_na.load_state_dict(best_state_na)
model_na.eval()
with torch.no_grad(), torch.amp.autocast('cuda'):
    tp, _, _ = model_na(edge_u, edge_indices_b, test_u, test_b)
    test_rmse_na = math.sqrt(F.mse_loss(tp, test_y).item())
print(f'\nNo Auxiliary Information')
print(f'  Best val RMSE:  {best_val_na:.4f} (epoch {best_epoch_na})')
print(f'  Test RMSE:      {test_rmse_na:.4f}')
del model_na; torch.cuda.empty_cache()

Ablation 4: No Auxiliary Information
Epoch   1 | Train: 1.5479 | Val: 1.6592 | lr=5.0e-03 <- best
Epoch   2 | Train: 1.5396 | Val: 1.6527 | lr=5.0e-03 <- best
Epoch   3 | Train: 1.5306 | Val: 1.6447 | lr=5.0e-03 <- best
Epoch   4 | Train: 1.5200 | Val: 1.6322 | lr=5.0e-03 <- best
Epoch   5 | Train: 1.5049 | Val: 1.6105 | lr=5.0e-03 <- best
Epoch   6 | Train: 1.4797 | Val: 1.5723 | lr=5.0e-03 <- best
Epoch   7 | Train: 1.4371 | Val: 1.5137 | lr=5.0e-03 <- best
Epoch   8 | Train: 1.3762 | Val: 1.4673 | lr=5.0e-03 <- best
Epoch   9 | Train: 1.3465 | Val: 1.4762 | lr=5.0e-03
Epoch  10 | Train: 1.3733 | Val: 1.4573 | lr=5.0e-03 <- best
Epoch  11 | Train: 1.3344 | Val: 1.4523 | lr=5.0e-03 <- best
Epoch  12 | Train: 1.3073 | Val: 1.4635 | lr=5.0e-03
Epoch  13 | Train: 1.3078 | Val: 1.4644 | lr=5.0e-03
Epoch  14 | Train: 1.3033 | Val: 1.4501 | lr=5.0e-03 <- best
Epoch  15 | Train: 1.2846 | Val: 1.4294 | lr=5.0e-03 <- best
Epoch  16 | Train: 1.2612 | Val: 1.4184 | lr=5.0e-03 <- best
Epoch  17 |

In [29]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ══════════════════════════════════════════════════════════════════════════
# Ablation 5: No Networks or Auxiliary Information
# Paper Table 2: pure matrix factorization — no graphs, no side features
# Only H4 embeddings + biases → dot product prediction
# ══════════════════════════════════════════════════════════════════════════

class PureMF(nn.Module):
    """Pure matrix factorization: H4_u · H4_b + biases, no graphs, no features."""
    def __init__(self, n_users, n_biz, kf=32, r_min=1.0, r_max=5.0):
        super().__init__()
        self.r_min, self.r_max = r_min, r_max
        self.H4_u = nn.Embedding(n_users, kf)
        self.H4_b = nn.Embedding(n_biz, kf)
        nn.init.normal_(self.H4_u.weight, std=0.01)
        nn.init.normal_(self.H4_b.weight, std=0.01)

        self.bias_u = nn.Embedding(n_users, 1)
        self.bias_b = nn.Embedding(n_biz, 1)
        self.bias_global = nn.Parameter(torch.zeros(1))
        nn.init.zeros_(self.bias_u.weight)
        nn.init.zeros_(self.bias_b.weight)

    def forward(self, user_idx, biz_idx):
        U_q = self.H4_u(user_idx)
        B_q = self.H4_b(biz_idx)
        logit = (U_q * B_q).sum(1) + self.bias_u(user_idx).squeeze(1) \
              + self.bias_b(biz_idx).squeeze(1) + self.bias_global
        pred = (self.r_max - self.r_min) * torch.sigmoid(logit) + self.r_min
        return pred


# ── Train ─────────────────────────────────────────────────────────────────
print('Ablation 5: No Networks or Auxiliary Information (Pure MF)')
print('=' * 60)

model_mf = PureMF(n_users, n_biz, kf=ABLATION_CONFIG['kf']).to(DEVICE)
opt_mf = Adam(model_mf.parameters(), lr=ABLATION_CONFIG['lr'], weight_decay=ABLATION_CONFIG['weight_decay'])
sched_mf = ReduceLROnPlateau(opt_mf, patience=5, factor=0.5)
scaler_mf = torch.amp.GradScaler('cuda')

best_val_mf, best_epoch_mf, no_imp_mf = float('inf'), 0, 0
best_state_mf = None

for epoch in range(1, ABLATION_CONFIG['epochs'] + 1):
    model_mf.train()
    opt_mf.zero_grad()
    with torch.amp.autocast('cuda'):
        pred = model_mf(train_u, train_b)
        mse = F.mse_loss(pred, train_y)
        # No graph regularization — no graphs
        loss = mse
    scaler_mf.scale(loss).backward()
    scaler_mf.unscale_(opt_mf)
    nn.utils.clip_grad_norm_(model_mf.parameters(), 5.0)
    scaler_mf.step(opt_mf)
    scaler_mf.update()

    model_mf.eval()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        vp = model_mf(val_u, val_b)
        val_rmse = math.sqrt(F.mse_loss(vp, val_y).item())
    sched_mf.step(val_rmse)

    flag = ''
    if val_rmse < best_val_mf:
        best_val_mf = val_rmse; best_epoch_mf = epoch; no_imp_mf = 0
        best_state_mf = {k: v.clone() for k, v in model_mf.state_dict().items()}
        flag = ' <- best'
    else:
        no_imp_mf += 1

    lr_now = opt_mf.param_groups[0]['lr']
    print(f'Epoch {epoch:3d} | Train: {math.sqrt(mse.item()):.4f} | Val: {val_rmse:.4f} | lr={lr_now:.1e}{flag}')
    if no_imp_mf >= ABLATION_CONFIG['patience']:
        print(f'Early stopping at epoch {epoch}'); break

model_mf.load_state_dict(best_state_mf)
model_mf.eval()
with torch.no_grad(), torch.amp.autocast('cuda'):
    tp = model_mf(test_u, test_b)
    test_rmse_mf = math.sqrt(F.mse_loss(tp, test_y).item())
print(f'\nNo Networks or Auxiliary Information (Pure MF)')
print(f'  Best val RMSE:  {best_val_mf:.4f} (epoch {best_epoch_mf})')
print(f'  Test RMSE:      {test_rmse_mf:.4f}')
del model_mf; torch.cuda.empty_cache()

Ablation 5: No Networks or Auxiliary Information (Pure MF)
Epoch   1 | Train: 1.5479 | Val: 1.6593 | lr=5.0e-03 <- best
Epoch   2 | Train: 1.5383 | Val: 1.6537 | lr=5.0e-03 <- best
Epoch   3 | Train: 1.5285 | Val: 1.6482 | lr=5.0e-03 <- best
Epoch   4 | Train: 1.5180 | Val: 1.6427 | lr=5.0e-03 <- best
Epoch   5 | Train: 1.5065 | Val: 1.6372 | lr=5.0e-03 <- best
Epoch   6 | Train: 1.4938 | Val: 1.6316 | lr=5.0e-03 <- best
Epoch   7 | Train: 1.4799 | Val: 1.6261 | lr=5.0e-03 <- best
Epoch   8 | Train: 1.4646 | Val: 1.6206 | lr=5.0e-03 <- best
Epoch   9 | Train: 1.4478 | Val: 1.6151 | lr=5.0e-03 <- best
Epoch  10 | Train: 1.4297 | Val: 1.6095 | lr=5.0e-03 <- best
Epoch  11 | Train: 1.4099 | Val: 1.6040 | lr=5.0e-03 <- best
Epoch  12 | Train: 1.3887 | Val: 1.5984 | lr=5.0e-03 <- best
Epoch  13 | Train: 1.3659 | Val: 1.5928 | lr=5.0e-03 <- best
Epoch  14 | Train: 1.3417 | Val: 1.5872 | lr=5.0e-03 <- best
Epoch  15 | Train: 1.3160 | Val: 1.5817 | lr=5.0e-03 <- best
Epoch  16 | Train: 1.2891 

In [30]:
# ══════════════════════════════════════════════════════════════════════════
# Full Ablation Summary (Table 2)
# ══════════════════════════════════════════════════════════════════════════
print('=' * 70)
print('Complete Ablation Study Summary (Paper Table 2)')
print('=' * 70)
print(f'{"Configuration":<45} {"Val RMSE":>10} {"Test RMSE":>10} {"Paper":>8}')
print('-' * 70)
print(f'{"Full MG-GAT":<45} {best_val_f:>10.4f} {test_rmse:>10.4f} {"1.249":>8}')
print(f'{"NIG Removed (uniform weights)":<45} {val_nig:>10.4f} {test_nig:>10.4f} {"1.303":>8}')
print(f'{"FR Removed (nonlinear Layer 1)":<45} {val_fr:>10.4f} {test_fr:>10.4f} {"1.305":>8}')
print(f'{"Uniform Graph Weighting":<45} {best_val_uw:>10.4f} {test_rmse_uw:>10.4f} {"1.280":>8}')
print(f'{"No Auxiliary Information":<45} {best_val_na:>10.4f} {test_rmse_na:>10.4f} {"1.312":>8}')
print(f'{"No Networks or Auxiliary Information":<45} {best_val_mf:>10.4f} {test_rmse_mf:>10.4f} {"1.405":>8}')
print('-' * 70)

# ── Save results to JSON ────────────────────────────────────────────────
import json as _json, os as _os
results = {
    'Full MG-GAT':                         {'val_rmse': round(best_val_f, 4),  'test_rmse': round(test_rmse, 4)},
    'NIG Removed (uniform weights)':       {'val_rmse': round(val_nig, 4),     'test_rmse': round(test_nig, 4)},
    'FR Removed (nonlinear Layer 1)':      {'val_rmse': round(val_fr, 4),      'test_rmse': round(test_fr, 4)},
    'Uniform Graph Weighting':             {'val_rmse': round(best_val_uw, 4), 'test_rmse': round(test_rmse_uw, 4)},
    'No Auxiliary Information':            {'val_rmse': round(best_val_na, 4), 'test_rmse': round(test_rmse_na, 4)},
    'No Networks or Auxiliary Information': {'val_rmse': round(best_val_mf, 4), 'test_rmse': round(test_rmse_mf, 4)},
}
_results_path = _os.path.join(WORK_DIR, 'ablation_results.json')
with open(_results_path, 'w') as _f:
    _json.dump(results, _f, indent=2)
print(f'\nResults saved to: {_results_path}')

Complete Ablation Study Summary (Paper Table 2)
Configuration                                   Val RMSE  Test RMSE    Paper
----------------------------------------------------------------------
Full MG-GAT                                       1.3891     1.4322    1.249
NIG Removed (uniform weights)                     1.3777     1.4226    1.303
FR Removed (nonlinear Layer 1)                    1.3896     1.4336    1.305
Uniform Graph Weighting                           1.3849     1.4292    1.280
No Auxiliary Information                          1.3880     1.4428    1.312
No Networks or Auxiliary Information              1.4049     1.4648    1.405
----------------------------------------------------------------------

Results saved to: /kaggle/working/ablation_results.json
